### Cargamos el dataset para evitar perder tiempo cargandolo varias veces.


In [13]:
cifar10 = tf.keras.datasets.cifar10

## Perceptrón multicapa utilizando varias compuertas

In [16]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

class LogicGateClassifier:
    """Clasificador de compuertas lógicas mediante un Perceptrón Multicapa."""

    def __init__(self, gate: str = 'XOR', hidden_units: int = 16, learning_rate: float = 0.01):
        self.gate = gate.upper()
        self.hidden_units = hidden_units
        self.learning_rate = learning_rate
        self.model = self._build_model()
        self.X, self.y = self._get_dataset()

    def _build_model(self) -> Sequential:
        model = Sequential([
            Input(shape=(2,)),
            Dense(self.hidden_units, activation='relu'),
            Dense(1, activation='sigmoid')
        ])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss='binary_crossentropy',
            metrics=['binary_accuracy']
        )
        return model

    def _get_dataset(self):
        X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
        if self.gate == 'XOR':
            y = np.array([[0], [1], [1], [0]], dtype=np.float32)
        elif self.gate == 'AND':
            y = np.array([[0], [0], [0], [1]], dtype=np.float32)
        elif self.gate == 'OR':
            y = np.array([[0], [1], [1], [1]], dtype=np.float32)
        else:
            raise ValueError("Compuerta no soportada. Usa 'XOR', 'AND' u 'OR'.")
        return X, y

    def train(self, epochs: int = 500, verbose: int = 0):
        print(f"\n--- Entrenando compuerta {self.gate} por {epochs} épocas ---")
        history = self.model.fit(self.X, self.y, epochs=epochs, verbose=verbose)
        return history

    def evaluate_and_display(self):
        predictions = self.model.predict(self.X, verbose=0)
        print(f"Resultados finales para compuerta {self.gate}:")
        for i, input_data in enumerate(self.X):
            prob = float(predictions[i][0])
            binary_class = int(round(prob))
            expected = int(self.y[i][0])
            print(f"Entrada: {input_data.astype(int)} -> Prob: {prob:.4f} | Predicha: {binary_class} | Esperada: {expected}")


# Prueba de las 3 compuertas
if __name__ == "__main__":
    for gate_type in ['AND', 'OR', 'XOR']:
        gate_net = LogicGateClassifier(gate=gate_type, hidden_units=8, learning_rate=0.05)
        gate_net.train(epochs=300)
        gate_net.evaluate_and_display()


--- Entrenando compuerta AND por 300 épocas ---
Resultados finales para compuerta AND:
Entrada: [0 0] -> Prob: 0.0000 | Predicha: 0 | Esperada: 0
Entrada: [0 1] -> Prob: 0.0143 | Predicha: 0 | Esperada: 0
Entrada: [1 0] -> Prob: 0.0003 | Predicha: 0 | Esperada: 0
Entrada: [1 1] -> Prob: 0.9990 | Predicha: 1 | Esperada: 1

--- Entrenando compuerta OR por 300 épocas ---
Resultados finales para compuerta OR:
Entrada: [0 0] -> Prob: 0.0189 | Predicha: 0 | Esperada: 0
Entrada: [0 1] -> Prob: 0.9998 | Predicha: 1 | Esperada: 1
Entrada: [1 0] -> Prob: 0.9997 | Predicha: 1 | Esperada: 1
Entrada: [1 1] -> Prob: 1.0000 | Predicha: 1 | Esperada: 1

--- Entrenando compuerta XOR por 300 épocas ---
Resultados finales para compuerta XOR:
Entrada: [0 0] -> Prob: 0.0052 | Predicha: 0 | Esperada: 0
Entrada: [0 1] -> Prob: 0.9994 | Predicha: 1 | Esperada: 1
Entrada: [1 0] -> Prob: 0.9993 | Predicha: 1 | Esperada: 1
Entrada: [1 1] -> Prob: 0.0052 | Predicha: 0 | Esperada: 0


## Clasificación de imagenes en blanco y negro

In [17]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Input, Dropout

class TunableFashionMNISTMLP:
    def __init__(
        self,
        input_shape: tuple = (28, 28),
        hidden_units: tuple = (128, 64),
        num_classes: int = 10,
        learning_rate: float = 0.001,
        epochs: int = 5,
        batch_size: int = 64,
        validation_split: float = 0.2,
        dropout_rate: float = 0.0
    ):
        self.input_shape = input_shape
        self.hidden_units = hidden_units
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.validation_split = validation_split
        self.dropout_rate = dropout_rate

        self.class_names = [
            'Camiseta', 'Pantalon', 'Sueter', 'Vestido', 'Abrigo',
            'Sandalia', 'Camisa', 'Tenis', 'Bolso', 'Botin'
        ]

        self.X_train, self.y_train = None, None
        self.X_test, self.y_test = None, None
        self.history = None
        self.model = self._build_model()

    def _build_model(self):
        layers = [Input(shape=self.input_shape), Flatten()]
        for units in self.hidden_units:
            layers.append(Dense(units, activation='relu'))
            if self.dropout_rate > 0.0:
                layers.append(Dropout(self.dropout_rate))

        layers.append(Dense(self.num_classes, activation='softmax'))

        model = Sequential(layers)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    def load_data(self):
        (X_tr, y_tr), (X_ts, y_ts) = tf.keras.datasets.fashion_mnist.load_data()
        self.X_train = X_tr / 255.0
        self.X_test = X_ts / 255.0
        self.y_train = y_tr
        self.y_test = y_ts

    def fit_and_evaluate(self):
        if self.X_train is None:
            self.load_data()

        self.history = self.model.fit(
            self.X_train, self.y_train,
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_split=self.validation_split,
            verbose=1
        )
        test_loss, test_acc = self.model.evaluate(self.X_test, self.y_test, verbose=0)
        return test_loss, test_acc

## Experimentación variando los parametros

In [18]:
experiments = [
    {"name": "Base original reducida", "hidden": (128, 64), "epochs": 5, "lr": 0.001},
    {"name": "1 Capa pequeña (Subajuste)", "hidden": (32,), "epochs": 3, "lr": 0.001},
    {"name": "1 Capa ancha", "hidden": (512,), "epochs": 5, "lr": 0.001},
    {"name": "Profunda (3 capas)", "hidden": (256, 128, 64), "epochs": 5, "lr": 0.001},
    {"name": "Profunda + Dropout", "hidden": (256, 128, 64), "epochs": 8, "lr": 0.0005, "dropout": 0.2}
]

for exp in experiments:
    print(f"\n================ Experimento: {exp['name']} ================")
    net = TunableFashionMNISTMLP(
        hidden_units=exp["hidden"],
        epochs=exp["epochs"],
        learning_rate=exp["lr"],
        dropout_rate=exp.get("dropout", 0.0)
    )
    t_loss, t_acc = net.fit_and_evaluate()
    print(f">> Resultado en Test -> Pérdida: {t_loss:.4f} | Precisión: {t_acc*100:.2f}%")


================ Experimento: Base original reducida ================
Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.8125 - loss: 0.5335 - val_accuracy: 0.8393 - val_loss: 0.4738
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.8602 - loss: 0.3898 - val_accuracy: 0.8635 - val_loss: 0.3716
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8737 - loss: 0.3470 - val_accuracy: 0.8695 - val_loss: 0.3587
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8823 - loss: 0.3207 - val_accuracy: 0.8803 - val_loss: 0.3326
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8892 - loss: 0.2999 - val_accuracy: 0.8782 - val_loss: 0.3391
>> Resultado en Test -> Pérdida: 0.3647 | Precisión: 87.30%

================ Experimento: 1 Capa pequeña (Subajuste) ================
Epoch 1/3
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7936 - loss: 0.6186 - val_accuracy: 0.8356 - val_loss: 0.4652
Epoch 2/3
750/750 ━━━━━━━━━━━━

## Clasificación de imagenes a color

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout, BatchNormalization, Input

class OptimizedColorMLP:
    def __init__(
        self,
        hidden_units=(1024, 512, 256, 128),
        epochs=25,
        batch_size=64,
        learning_rate=0.0005,
        dropout_rate=0.25
    ):
        self.hidden_units = hidden_units
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.dropout_rate = dropout_rate
        self.model = self._build_model()

    def _build_model(self):
        layers = [Input(shape=(32, 32, 3)), Flatten()]

        for units in self.hidden_units:
            layers.append(Dense(units, activation='relu'))
            layers.append(BatchNormalization())
            layers.append(Dropout(self.dropout_rate))

        layers.append(Dense(10, activation='softmax'))

        model = Sequential(layers)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    def train_and_evaluate(self):
        (X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
        X_train, X_test = X_train / 255.0, X_test / 255.0
        y_train, y_test = y_train.flatten(), y_test.flatten()

        print("Entrenando MLP optimizado para CIFAR-10...")
        history = self.model.fit(
            X_train, y_train,
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_split=0.2,
            verbose=1
        )

        loss, acc = self.model.evaluate(X_test, y_test, verbose=0)
        print(f"\nExactitud final en Test (CIFAR-10): {acc * 100:.2f}% (Pérdida: {loss:.4f})")
        return history, acc

# Ejecución
cifar_mlp = OptimizedColorMLP(epochs=20, learning_rate=0.0003)
history, final_acc = cifar_mlp.train_and_evaluate()

Entrenando MLP optimizado para CIFAR-10...
Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 76ms/step - accuracy: 0.2917 - loss: 2.1219 - val_accuracy: 0.3470 - val_loss: 1.8467
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.3539 - loss: 1.8338 - val_accuracy: 0.3151 - val_loss: 1.9279
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 49s 78ms/step - accuracy: 0.3829 - loss: 1.7245 - val_accuracy: 0.1881 - val_loss: 2.6724
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 81s 77ms/step - accuracy: 0.4017 - loss: 1.6678 - val_accuracy: 0.4146 - val_loss: 1.6438
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 50s 80ms/step - accuracy: 0.4180 - loss: 1.6253 - val_accuracy: 0.3966 - val_loss: 1.6831
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 49s 78ms/step - accuracy: 0.4257 - loss: 1.5998 - val_accuracy: 0.4565 - val_loss: 1.5164
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 48s 77ms/step - accuracy: 0.4367 - loss: 1.5733 - val_accuracy: 0.4223 - val_loss: 1.6020
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 48s 77m